In [7]:
import google.genai as genai
import google.genai.types as types
from dotenv import load_dotenv
import os
import IPython.display as ipd
import wave
import struct

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [8]:
response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents="Hello world! This is a TTS test.",
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Aoede")
            )
        )
    )
)

# mp3로 저장
audio_data = response.candidates[0].content.parts[0].inline_data.data
with wave.open("hello_world.wav", "wb") as wav_file:
    wav_file.setnchannels(1)      # 모노
    wav_file.setsampwidth(2)      # 16bit
    wav_file.setframerate(24000)  # 24kHz
    wav_file.writeframes(audio_data)

ipd.Audio("hello_world.wav")

In [10]:
voice = "Charon"  # Gemini 목소리
wav_file = f"hello_world_{voice}.wav"

response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents=f"Hello world! I'm {voice}. This is a TTS test.",
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice)
            )
        )
    )
)

audio_data = response.candidates[0].content.parts[0].inline_data.data

with wave.open(wav_file, "wb") as f:
    f.setnchannels(1)
    f.setsampwidth(2)
    f.setframerate(24000)
    f.writeframes(audio_data)

ipd.Audio(wav_file)


In [11]:
import json

# json 파일 열기
with open('./data/image_quiz_eng.json', 'r', encoding='utf-8') as f:
    eng_dict = json.load(f)

eng_dict


[{'no': 1,
  'eng': 'Which of the following descriptions of the image is incorrect?\n- (1) A lion is leaping through the air to catch a gazelle.\n- (2) The gazelle is running in the opposite direction to get away from the lion.\n- (3) The lion in the image is a lioness without a mane.\n- (4) A wide field and grass can be seen in the background.',
  'img': '360_F_1076467513_QaPdOB3I0SqhkjN7HRvjFvhTPN8zsN5c.jpg'},
 {'no': 2,
  'eng': "Which of the following descriptions of the image is incorrect?\n- (1) It captures a tense moment where a tiger is pouncing toward a deer.\n- (2) Large and magnificent antlers are growing on the deer's head.\n- (3) The background is a plain, and dry grass is growing on the ground.\n- (4) The tiger is opening its mouth wide and extending its sharp front paws to catch the deer.\n\n**",
  'img': '360_F_1537162917_C20wiVgwVtv1klja53NBIltckHJTAr5p.jpg'},
 {'no': 3,
  'eng': 'Which of the following descriptions of the image is incorrect?\n- (1) A large screen disp

In [14]:
voices = ["Aoede", "Charon", "Fenrir", "Kore", "Puck", "Leda", "Orus", "Zephyr"]

for q in eng_dict:
    no = q['no']
    quiz = q['eng']
    quiz = quiz.replace("- (1)", "- One.\t")
    quiz = quiz.replace("- (2)", "- Two.\t")
    quiz = quiz.replace("- (3)", "- Three.\t")
    quiz = quiz.replace("- (4)", "- Four.\t")

    print(no, quiz)

    voice = voices[no % len(voices)]

    for attempt in range(3):  # 최대 3번 재시도
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash-preview-tts",
                contents=f'#{no}. {quiz}',
                config=types.GenerateContentConfig(
                    response_modalities=["AUDIO"],
                    speech_config=types.SpeechConfig(
                        voice_config=types.VoiceConfig(
                            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice)
                        )
                    )
                )
            )
            audio_data = response.candidates[0].content.parts[0].inline_data.data
            with wave.open(rf'.\audio\{no}.wav', 'wb') as f:
                f.setnchannels(1)
                f.setsampwidth(2)
                f.setframerate(24000)
                f.writeframes(audio_data)
            print(f"{no}번 완료")
            break  # 성공하면 재시도 루프 탈출

        except Exception as e:
            print(f"{no}번 실패 ({attempt+1}/3): {e}")
            time.sleep(5)  # 5초 기다렸다가 재시도

1 Which of the following descriptions of the image is incorrect?
- One.	 A lion is leaping through the air to catch a gazelle.
- Two.	 The gazelle is running in the opposite direction to get away from the lion.
- Three.	 The lion in the image is a lioness without a mane.
- Four.	 A wide field and grass can be seen in the background.
1번 완료
2 Which of the following descriptions of the image is incorrect?
- One.	 It captures a tense moment where a tiger is pouncing toward a deer.
- Two.	 Large and magnificent antlers are growing on the deer's head.
- Three.	 The background is a plain, and dry grass is growing on the ground.
- Four.	 The tiger is opening its mouth wide and extending its sharp front paws to catch the deer.

**
2번 완료
3 Which of the following descriptions of the image is incorrect?
- One.	 A large screen displays the text "DIVE 2024 IN BUSAN."
- Two.	 Many people are sitting at tables working on their laptops.
- Three.	 Pink signs labeled "REFRESH ZONE" can be seen along the 